# Whisper-small Hindi — LoRA fine-tuning (Colab T4)

One-click end-to-end run: **data → features → LoRA train → evaluate → plot → (optional) push**.

**Before running:** `Runtime → Change runtime type → T4 GPU`.

All hyperparameters live in `configs/whisper_small_hi.yaml`.

In [ ]:
# 1. Confirm we have a GPU (expect a Tesla T4).
!nvidia-smi

In [ ]:
# 2. Get the project code.
# Option A: clone from GitHub (replace with your repo URL).
REPO_URL = "https://github.com/<your-username>/whisper_finetune.git"

import os
if not os.path.isdir("whisper_finetune"):
    !git clone $REPO_URL
%cd whisper_finetune

# Option B: if you uploaded the folder manually instead, just `%cd` into it.

In [ ]:
# 3. Install dependencies.
# IMPORTANT: do NOT reinstall torch on Colab — it ships a CUDA-matched build.
# We install everything else at the pinned versions.
!pip install -q \
    transformers==4.44.2 \
    peft==0.12.0 \
    datasets==2.21.0 \
    accelerate==0.33.0 \
    evaluate==0.4.2 \
    jiwer==3.0.4 \
    librosa==0.10.2 \
    soundfile==0.12.1 \
    huggingface_hub==0.24.6 \
    PyYAML==6.0.2
print('deps installed')

In [ ]:
# 4. Log in to the Hugging Face Hub.
# Common Voice 17 is GATED: accept its terms on the dataset page first, then
# paste a token (with read access) here.
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# 5. (Optional) Warm the dataset cache and print split stats.
!python scripts/prepare_data.py --config configs/whisper_small_hi.yaml

In [ ]:
# 6. Train. Prints the LoRA trainable-parameter % (expect < 2%) at startup.
# On a T4 this runs in fp16 automatically. Trim epochs in the config if you're
# close to the 12h session limit.
!python scripts/run_train.py --config configs/whisper_small_hi.yaml

In [ ]:
# 7. Evaluate baseline vs fine-tuned on the test split.
!python scripts/run_eval.py \
    --config configs/whisper_small_hi.yaml \
    --adapter artifacts/whisper-small-hi-lora \
    --output artifacts/eval_results.json

In [ ]:
# 8. Plot the comparison and display it inline.
!python scripts/plot_results.py \
    --results artifacts/eval_results.json \
    --output artifacts/wer_cer_comparison.png

from IPython.display import Image
Image('artifacts/wer_cer_comparison.png')

## 9. (Optional) Push the adapter to the Hub

Set `hub.push_to_hub: true` and `hub.repo_id` in `configs/whisper_small_hi.yaml` before
training, **or** push manually below. The adapter is only a few MB.

In [ ]:
REPO_ID = "<your-username>/whisper-small-hi-lora"

from peft import PeftModel
from transformers import WhisperForConditionalGeneration, WhisperProcessor

base = WhisperForConditionalGeneration.from_pretrained('openai/whisper-small')
model = PeftModel.from_pretrained(base, 'artifacts/whisper-small-hi-lora')
processor = WhisperProcessor.from_pretrained('artifacts/whisper-small-hi-lora')

model.push_to_hub(REPO_ID)
processor.push_to_hub(REPO_ID)
print(f'pushed to https://huggingface.co/{REPO_ID}')